# DeepSpeed-MII: Low-Latency, High-Throughput Model Inference

**DeepSpeed-MII** (Model Implementations for Inference) is an open-source Python
library from Microsoft's DeepSpeed team that makes low-latency, low-cost
inference of large language models both feasible and easy. It wraps the
DeepSpeed-Inference engine behind a few lines of Python, applying kernel
fusion, tensor parallelism, and (since MII 0.1) a high-throughput serving
backend called **DeepSpeed-FastGen** built on *Dynamic SplitFuse* batching.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

### What is it?

DeepSpeed-MII is an inference-serving library that sits on top of the DeepSpeed
runtime. It exposes two pipelines:

- **MII Serverless / `pipeline`** — the original non-persistent path for offline
  or single-process batch generation.
- **MII Persistent deployment (`mii.serve`)** — a persistent, gRPC-backed model
  server powered by **DeepSpeed-FastGen**, designed for production serving with
  continuous batching and tensor parallelism.

The core technology underneath is **Dynamic SplitFuse**: instead of running the
compute-heavy prompt ("prefill") phase and the latency-sensitive token-by-token
("decode") phase in separate batches, FastGen splits long prompts into chunks
and *fuses* prompt chunks with ongoing decode tokens into uniformly sized
forward passes. This keeps the GPU saturated, smooths tail latency, and raises
effective throughput.

### Why use it?

- **High throughput** — DeepSpeed-FastGen reports up to ~2.3x higher effective
  throughput and lower tail latency than vLLM on long-prompt workloads, thanks
  to Dynamic SplitFuse batching.
- **Low latency** — fused CUDA kernels (attention, GeLU, LayerNorm) and a
  blocked KV cache reduce time-to-first-token and per-token latency.
- **Simple API** — `mii.pipeline()` or `mii.serve()` deploys a model in a couple
  of lines; an OpenAI-compatible REST gateway and a gRPC client are included.
- **Scales across GPUs** — tensor parallelism is a single `tensor_parallel`
  argument; replicas can be load-balanced behind one endpoint.

### When to use it?

- You serve open transformer LLMs (Llama, Mistral, Falcon, OPT, Qwen, Phi) and
  want maximum tokens/sec per GPU dollar.
- Your traffic mixes long prompts with streaming chat generation — the workload
  Dynamic SplitFuse is specifically tuned for.
- You are already in the DeepSpeed/PyTorch ecosystem and want inference without
  switching frameworks.

## Key Features

| Feature | Description | Benefit |
|---------|-------------|---------|
| Dynamic SplitFuse | Chunks long prompts and fuses prefill with decode into uniform forward passes | Higher GPU utilization, lower tail latency |
| Blocked KV cache | Paged, non-contiguous KV-cache blocks (similar in spirit to PagedAttention) | Less memory fragmentation, larger batch sizes |
| Fused CUDA kernels | Custom kernels for attention, MLP, LayerNorm, and sampling | Faster per-token generation |
| Persistent deployment | `mii.serve` launches a gRPC server with load-balanced replicas | Production-grade serving with auto-batching |
| Tensor parallelism | `tensor_parallel=N` shards a model across N GPUs | Serve models larger than a single GPU |
| OpenAI-compatible API | `mii.serve(..., enable_restful_api=True)` exposes `/v1` endpoints | Drop-in for existing OpenAI clients |
| Non-persistent pipeline | `mii.pipeline()` for offline/batch generation in one process | Easy benchmarking and batch jobs |

## Architecture Overview

```
                ┌──────────────────────────────────────────┐
   HTTP / gRPC  │            Clients (chat, batch)          │
  ────────────► └───────────────────┬──────────────────────┘
                                    │
                    ┌───────────────▼────────────────┐
                    │   RESTful Gateway (optional)    │  OpenAI-compatible /v1
                    └───────────────┬────────────────┘
                                    │
                    ┌───────────────▼────────────────┐
                    │      MII Load Balancer (gRPC)   │  routes to replicas
                    └───────────────┬────────────────┘
                       ┌────────────┼────────────┐
                       ▼            ▼             ▼
                 ┌──────────┐ ┌──────────┐  ┌──────────┐
                 │ Replica0 │ │ Replica1 │  │ ReplicaN │   each = model engine
                 └────┬─────┘ └────┬─────┘  └────┬─────┘
                      │            │             │
            ┌─────────▼────────────▼─────────────▼─────────┐
            │        DeepSpeed-FastGen Inference Engine     │
            │  • Dynamic SplitFuse scheduler                │
            │  • Blocked KV cache manager                   │
            │  • Fused CUDA kernels (attn / MLP / sampling) │
            │  • Tensor parallelism across GPUs             │
            └───────────────────────────────────────────────┘
```

### Components

1. **RESTful Gateway** — optional FastAPI-style front end exposing
   OpenAI-compatible `/v1/completions` and `/v1/chat/completions`.
2. **Load Balancer** — a gRPC server process that fans requests out across model
   replicas and aggregates streamed responses.
3. **Model Replica(s)** — each replica owns a DeepSpeed-FastGen engine and, when
   `tensor_parallel > 1`, spans multiple GPUs.
4. **FastGen Engine** — the inference core: the Dynamic SplitFuse scheduler, the
   blocked KV-cache manager, and the fused CUDA kernels.

## Installation

### Prerequisites

- **Python** 3.8+ (3.9–3.11 recommended)
- **NVIDIA GPU** with CUDA 11.6+ and a matching PyTorch build
- **PyTorch** 2.0+ installed before DeepSpeed so the CUDA ops build correctly
- A C++ compiler / CUDA toolkit for the DeepSpeed JIT kernels (`ninja` helps)

### Installation Steps

`deepspeed-mii` pulls in `deepspeed` and `transformers` automatically. The
package name on PyPI is **`deepspeed-mii`** (hyphen), imported as `mii`.

**Note**: Uncomment the cell below to install (e.g. in Google Colab with a GPU
runtime). MII requires a CUDA GPU; it will not run on CPU-only environments.

In [ ]:
# Uncomment to install DeepSpeed-MII (requires a CUDA GPU)
# !pip install deepspeed-mii

# Verify the install and inspect the build environment:
# !python -c "import mii, deepspeed; print('mii', mii.__version__, '| deepspeed', deepspeed.__version__)"
# !ds_report  # prints which DeepSpeed ops are compiled/compatible

## Basic Usage

### Quick Start: the non-persistent pipeline

`mii.pipeline()` loads a model into the current process and returns a callable.
This is the simplest way to generate text and is ideal for offline batch jobs
and benchmarking.

In [ ]:
import mii

# Load a model into a one-process FastGen pipeline.
# The model id is any HuggingFace causal-LM checkpoint.
pipe = mii.pipeline("mistralai/Mistral-7B-Instruct-v0.2")

prompts = [
    "Explain tensor parallelism in one sentence.",
    "Write a haiku about GPUs.",
]

# Generation kwargs map onto FastGen's sampling config.
responses = pipe(prompts, max_new_tokens=128, temperature=0.7, top_p=0.95)

for prompt, resp in zip(prompts, responses):
    print(f"PROMPT: {prompt}")
    print(f"OUTPUT: {resp.generated_text}\n")

# Free GPU memory when done (important inside notebooks).
pipe.destroy()

### Persistent deployment: `mii.serve`

For production you want a persistent server that keeps the model resident and
batches incoming requests. `mii.serve()` starts the gRPC load balancer plus the
replica(s); `mii.client()` connects to it from anywhere.

In [ ]:
import mii

# Start a persistent deployment. This launches background server processes
# and returns immediately. `deployment_name` identifies the endpoint.
client = mii.serve(
    "mistralai/Mistral-7B-Instruct-v0.2",
    deployment_name="mistral-deploy",
    tensor_parallel=1,     # set to the number of GPUs to shard across
    replica_num=1,         # number of independent model replicas
)

# Query the running deployment.
response = client.generate(
    ["List three benefits of continuous batching."],
    max_new_tokens=200,
    temperature=0.6,
)
print(response[0].generated_text)

# Connect from a separate process/notebook without re-serving:
# client = mii.client("mistral-deploy")

# Shut the deployment down (stops the background processes).
client.terminate_server()

### OpenAI-compatible REST endpoint

Pass `enable_restful_api=True` (and a port) to expose an OpenAI-style HTTP API,
so existing OpenAI SDK code can target MII by changing only the `base_url`.

In [ ]:
import mii

client = mii.serve(
    "mistralai/Mistral-7B-Instruct-v0.2",
    deployment_name="mistral-rest",
    enable_restful_api=True,
    restful_api_port=28080,
)

# In another process you can now call the REST gateway, e.g.:
#
#   curl -X POST http://localhost:28080/v1/chat/completions \
#     -H 'Content-Type: application/json' \
#     -d '{"model": "mistral-rest",
#          "messages": [{"role": "user", "content": "Hello!"}],
#          "max_tokens": 64}'
#
# Or with the OpenAI Python SDK:
#   from openai import OpenAI
#   oai = OpenAI(base_url="http://localhost:28080/v1", api_key="not-needed")
#   oai.chat.completions.create(model="mistral-rest",
#       messages=[{"role": "user", "content": "Hello!"}])

client.terminate_server()

## Advanced Features

### Streaming token-by-token output

Pass a callback to `client.generate(..., streaming_fn=...)` to receive tokens as
they are produced — essential for responsive chat UIs.

In [ ]:
import mii

client = mii.serve("mistralai/Mistral-7B-Instruct-v0.2",
                   deployment_name="stream-demo")

def on_token(response_chunk):
    # Each chunk carries newly generated text for one request.
    print(response_chunk[0].generated_text, end="", flush=True)

client.generate(
    ["Tell me a short story about a robot learning to paint."],
    max_new_tokens=256,
    streaming_fn=on_token,
)
print()
client.terminate_server()

### Multi-GPU tensor parallelism and replicas

Two orthogonal knobs scale a deployment:

- **`tensor_parallel`** shards a *single* model across N GPUs so you can serve
  models too large for one device (e.g. a 70B model across 4 GPUs).
- **`replica_num`** creates independent copies of the model, each load-balanced,
  to raise total request throughput.

In [ ]:
import mii

# Serve a 70B model sharded across 4 GPUs, with 2 such replicas (8 GPUs total).
client = mii.serve(
    "meta-llama/Meta-Llama-3-70B-Instruct",
    deployment_name="llama70b",
    tensor_parallel=4,   # shard one model over 4 GPUs
    replica_num=2,       # two replicas behind the load balancer
)

out = client.generate(["Summarize the theory of relativity for a 10-year-old."],
                      max_new_tokens=180)
print(out[0].generated_text)
client.terminate_server()

### Tuning generation and the KV cache

`mii.serve` accepts a `ModelConfig` (or keyword overrides) controlling the
FastGen scheduler. The most impactful fields:

- `max_length` — maximum sequence length (prompt + generation) the engine plans
  KV-cache blocks for.
- `max_ragged_batch_size` — the cap on how many requests SplitFuse fuses into a
  single forward pass.
- `quantization_mode` — optional weight quantization to fit larger models.

In [ ]:
import mii
from mii.config import ModelConfig

config = ModelConfig(
    model_name_or_path="mistralai/Mistral-7B-Instruct-v0.2",
    tensor_parallel=1,
    max_length=4096,            # plan KV cache for up to 4k tokens
    replica_num=1,
)

client = mii.serve(config.model_name_or_path,
                   deployment_name="tuned",
                   model_config=config)
print(client.generate(["ping"], max_new_tokens=8)[0].generated_text)
client.terminate_server()

## Use Cases

#### Use Case 1: High-throughput chat backend

- **Context**: A multi-tenant chat product serving thousands of concurrent
  sessions with variable prompt lengths.
- **Implementation**: `mii.serve` with `replica_num` scaled to the GPU count and
  the REST gateway enabled; Dynamic SplitFuse fuses the long system prompts with
  active decode steps to keep latency stable under load.
- **Results**: Higher sustained tokens/sec and tighter p95 latency than static
  batching, because no replica idles waiting for a batch to fill.

#### Use Case 2: Offline batch generation / evaluation

- **Context**: Generating synthetic data or scoring a large evaluation set.
- **Implementation**: `mii.pipeline()` over a list of prompts in a single
  process — no server lifecycle to manage.
- **Results**: Maximum throughput per GPU with the simplest possible code path.

#### Use Case 3: Serving a model too large for one GPU

- **Context**: A 70B-parameter model that does not fit in 80 GB.
- **Implementation**: `tensor_parallel=4` shards the weights and KV cache across
  four GPUs transparently.
- **Results**: The model serves at interactive latency without CPU offloading.

## Best Practices

1. **Match PyTorch/CUDA versions to your DeepSpeed build.** Install PyTorch
   first, then `deepspeed-mii`, and run `ds_report` to confirm the inference
   kernels compiled. Mismatches are the #1 source of install failures.
2. **Use the persistent deployment for serving, the pipeline for batch jobs.**
   `mii.serve` keeps the model resident and batches; `mii.pipeline` is for
   one-shot/offline work and benchmarking.
3. **Set `max_length` to your real workload, not the model maximum.** KV-cache
   blocks are pre-planned from `max_length`; an oversized value wastes GPU
   memory and shrinks the batch you can hold.
4. **Scale throughput with `replica_num`, capacity with `tensor_parallel`.**
   Add replicas for more concurrent requests; add tensor parallelism only when
   the model does not fit (or to cut per-token latency on a single large model).
5. **Always `terminate_server()` / `destroy()`.** Orphaned MII processes hold
   GPU memory; clean them up explicitly, especially in notebooks and tests.
6. **Pin the model into a warm deployment before traffic.** The first request
   pays JIT-kernel compilation and weight-load cost; send a warm-up request.

## Common Pitfalls

1. **Installing on a CPU-only machine.** MII needs a CUDA GPU; the engine will
   fail to build/initialize without one. Use a GPU runtime.
2. **Forgetting to terminate deployments.** Each `mii.serve` spawns persistent
   background processes that survive your script. Call `terminate_server()` (or
   `mii.client(name).terminate_server()`) or they leak GPU memory.
3. **Confusing `tensor_parallel` with `replica_num`.** Tensor parallel shards
   *one* model (needs enough GPUs for the shard count); replicas duplicate the
   model. Setting `tensor_parallel` higher than your GPU count fails to launch.
4. **Setting `max_length` too high.** Over-provisioning the KV cache reduces the
   number of concurrent sequences and can trigger out-of-memory at load time.
5. **Expecting every architecture to be supported.** FastGen covers the common
   decoder-only families (Llama, Mistral, Falcon, OPT, Qwen, Phi, MPT). Exotic
   or very new architectures may fall back or be unsupported — check the docs.

## Performance Optimization

### Configuration Tuning

Key parameters to optimize:

- **`max_ragged_batch_size`** — raise it to let SplitFuse fuse more requests per
  forward pass (more throughput) until you hit a latency or memory ceiling.
- **`tensor_parallel`** — increase to cut single-request latency on large models
  by spreading compute across GPUs (with communication overhead).
- **`replica_num`** — increase to serve more concurrent requests; throughput
  scales near-linearly with replicas while each stays latency-stable.
- **`max_length`** — set as low as your workload allows to free KV-cache memory
  for larger effective batches.
- **`quantization_mode`** — apply weight quantization to fit bigger models or
  larger batches on the same hardware.

The cell below benchmarks effective throughput so you can compare settings.

In [ ]:
import time
import mii

client = mii.serve("mistralai/Mistral-7B-Instruct-v0.2",
                   deployment_name="bench")

prompts = ["Write a paragraph about renewable energy."] * 64
max_new = 128

# Warm up (pays kernel-compile + weight-load once).
client.generate(["warmup"], max_new_tokens=8)

start = time.time()
responses = client.generate(prompts, max_new_tokens=max_new)
elapsed = time.time() - start

gen_tokens = sum(len(r.generated_text.split()) for r in responses)  # rough proxy
print(f"Requests: {len(prompts)}  | wall time: {elapsed:.2f}s")
print(f"Approx throughput: {gen_tokens / elapsed:.1f} words/s")
print(f"Latency per request: {elapsed / len(prompts):.3f}s")

client.terminate_server()

## Production Deployment

### Docker Deployment

Build on a CUDA-enabled base image so the DeepSpeed kernels can compile and run.

```dockerfile
FROM nvidia/cuda:12.1.1-cudnn8-runtime-ubuntu22.04

RUN apt-get update && apt-get install -y python3 python3-pip git && \
    rm -rf /var/lib/apt/lists/*

# Install a CUDA-matched PyTorch first, then MII.
RUN pip3 install torch --index-url https://download.pytorch.org/whl/cu121 && \
    pip3 install deepspeed-mii

COPY serve.py /app/serve.py
WORKDIR /app

EXPOSE 28080
# serve.py calls mii.serve(..., enable_restful_api=True, restful_api_port=28080)
CMD ["python3", "serve.py"]
```

Run with GPU access: `docker run --gpus all -p 28080:28080 my-mii-image`.

### Kubernetes Deployment

Request a GPU and front the REST gateway with a Service.

```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: mii-mistral
spec:
  replicas: 1
  selector:
    matchLabels:
      app: mii-mistral
  template:
    metadata:
      labels:
        app: mii-mistral
    spec:
      containers:
      - name: mii
        image: my-registry/mii-mistral:latest
        ports:
        - containerPort: 28080
        resources:
          limits:
            nvidia.com/gpu: 1
        readinessProbe:
          httpGet:
            path: /v1/models
            port: 28080
          initialDelaySeconds: 120   # allow weight load + kernel compile
          periodSeconds: 15
---
apiVersion: v1
kind: Service
metadata:
  name: mii-mistral
spec:
  selector:
    app: mii-mistral
  ports:
  - port: 80
    targetPort: 28080
```

## Monitoring and Observability

#### Key Metrics to Track

- **Time-to-first-token (TTFT)** — user-perceived responsiveness; rises when the
  prefill queue backs up. Watch it to size `replica_num`.
- **Per-token latency / inter-token latency** — steady-state generation speed;
  degradation signals GPU saturation or oversized batches.
- **Throughput (tokens/sec)** — aggregate generated tokens across replicas.
- **GPU utilization & memory** — from `nvidia-smi` / DCGM; near-100% util with
  headroom in memory is the target operating point.
- **Queue depth / pending requests** — a leading indicator that you need more
  replicas or that `max_ragged_batch_size` is too small.

#### Logging Best Practices

- Log request id, prompt/length, generated-token count, TTFT, and total latency
  per request as structured JSON.
- Export GPU metrics with the NVIDIA DCGM exporter and scrape with Prometheus;
  visualize TTFT and throughput in Grafana.
- Use appropriate log levels: DEBUG for token-level traces only in development;
  INFO for request summaries in production.

## Troubleshooting

#### Issue 1: `ds_report` shows inference ops as not compatible / build errors

**Symptoms**: Import works but `mii.serve` fails compiling CUDA ops, or you see
"Torch CUDA version mismatch".

**Cause**: PyTorch was built against a different CUDA toolkit than the one on the
machine, or no compiler/CUDA dev headers are present.

**Solution**: Install a PyTorch wheel matching your CUDA version *first*, ensure
`nvcc`/build tools are available, install `ninja`, then reinstall
`deepspeed-mii`. Confirm with `ds_report`.

#### Issue 2: CUDA out-of-memory at deployment or under load

**Symptoms**: `torch.cuda.OutOfMemoryError` during `mii.serve` or when batches
grow.

**Cause**: `max_length` over-provisions the KV cache, the model is too large for
the GPU, or too many replicas share one device.

**Solution**: Lower `max_length` to your real maximum, enable
`quantization_mode`, increase `tensor_parallel` to shard the model, or reduce
`replica_num` per GPU.

#### Issue 3: Orphaned processes hold the GPU after a crash

**Symptoms**: "address already in use" on the gRPC port, or `nvidia-smi` shows
memory used with no active script.

**Cause**: A previous `mii.serve` deployment was not terminated.

**Solution**: Call `mii.client(name).terminate_server()`, or kill the stray
python processes shown in `nvidia-smi`, then redeploy on a fresh port.

## Comparison with Alternatives

| Feature | DeepSpeed-MII | vLLM | TGI (Text Generation Inference) |
|---------|---------------|------|---------------------------------|
| Core batching | Dynamic SplitFuse (chunked prefill + fused decode) | Continuous batching + PagedAttention | Continuous batching |
| KV cache | Blocked / paged KV cache | PagedAttention | Paged attention |
| Multi-GPU | Tensor parallel + replicas | Tensor (+ pipeline) parallel | Tensor parallel (sharding) |
| API | gRPC client + OpenAI-compatible REST | OpenAI-compatible REST | REST + OpenAI-compatible |
| Ecosystem | DeepSpeed / PyTorch | Standalone, very large community | Hugging Face |
| Best at | Long-prompt + decode mixes, DeepSpeed users | General-purpose, broadest model support | Tight HF integration |

### When to Choose DeepSpeed-MII

- Your workload mixes **long prompts with streaming generation** — the case
  Dynamic SplitFuse optimizes for tail latency.
- You are **already invested in DeepSpeed/PyTorch** and want inference without a
  new framework.
- You want a **single library** that gives both an offline pipeline and a
  production gRPC/REST server with tensor parallelism and replicas.

## Resources

### Official Documentation

- DeepSpeed-MII GitHub: https://github.com/deepspeedai/DeepSpeed-MII
- DeepSpeed project site: https://www.deepspeed.ai/
- DeepSpeed-FastGen blog (Dynamic SplitFuse): https://github.com/deepspeedai/DeepSpeed/tree/master/blogs/deepspeed-fastgen

### Tutorials and Guides

- MII README & examples: https://github.com/deepspeedai/DeepSpeed-MII#readme
- FastGen deep-dive / benchmarks: https://github.com/deepspeedai/DeepSpeed/tree/master/blogs/deepspeed-fastgen
- DeepSpeed inference tutorial: https://www.deepspeed.ai/tutorials/inference-tutorial/

### Community Resources

- GitHub Issues / Discussions: https://github.com/deepspeedai/DeepSpeed-MII/issues
- DeepSpeed Discord / community channels (linked from deepspeed.ai)
- Stack Overflow tag: `deepspeed`

### Related Technologies

- **DeepSpeed-Inference** — the underlying optimized inference runtime
- **vLLM** — PagedAttention-based serving engine
- **TGI** — Hugging Face Text Generation Inference
- **TensorRT-LLM** — NVIDIA's optimized LLM inference compiler/runtime